# Lab 4 多模態 AI：讓模型「看圖」讀財報

Lab 2、Lab 3 的模型都只讀**文字**。今天讓它**看圖**——直接讀 K 線圖、損益表截圖，並拿**開源模型 `minimax-m3:cloud`** 和**商用旗艦 `gemini-3.5-flash`** 讀同一張圖，比較誰讀得準、誰比較貴。

> 📄 圖檔放在 `data/`（先進光 K 線、台積電損益表，皆為公開圖表，僅供讀圖練習）。

In [ ]:
# 📦 先跑這一格：一次裝齊今天要用的套件（指定版本，避免學校電腦裝到不相容的舊版；裝不起來看 README）
!pip install -q langchain-ollama==1.0.1 langchain-google-genai==4.2.7 langchain-core==1.4.9 requests

## 🔧 第 0 步：環境自我檢查（三格，跑一格看一個燈）

In [ ]:
# ✅ 檢查 1：設定兩把 API key（呼叫雲端模型的「門票」，缺一個今天都做不下去）
import os

# 👇 把引號中間換成你自己的 key（課前各自辦好的兩把，前後別留空白）
os.environ["OLLAMA_API_KEY"] = "在這裡貼上你的 OLLAMA key"
os.environ["GOOGLE_API_KEY"] = "在這裡貼上你的 Gemini key"

ollama_key = os.environ.get("OLLAMA_API_KEY")
google_key = os.environ.get("GOOGLE_API_KEY")
ollama_ok = ollama_key and ollama_key != "在這裡貼上你的 OLLAMA key"
google_ok = google_key and google_key != "在這裡貼上你的 Gemini key"
if ollama_ok and google_ok:
    print("✅ 兩把 key 都設定好了")
else:
    if not ollama_ok:
        print("❌ 還沒貼 OLLAMA key")
    if not google_ok:
        print("❌ 還沒貼 Gemini key（https://aistudio.google.com/apikey 免費辦）")

# 🔒 key 等於你帳號的鑰匙：貼了 key 的 notebook 別上傳 GitHub、別傳給別人、別截圖。

In [ ]:
# ✅ 檢查 2：套件裝了沒
try:
    from langchain_ollama import ChatOllama
    from langchain_google_genai import ChatGoogleGenerativeAI
    from langchain_core.messages import HumanMessage
    print("✅ 套件都在")
except Exception as e:
    print("❌ 套件缺東西 →", e)
    print("   → 終端機下：pip install -r requirements.txt")

In [ ]:
# ✅ 檢查 3：雲端 minimax 通不通（先用純文字問一句，確認門票有效）
from langchain_ollama import ChatOllama

ping = ChatOllama(
    model="minimax-m3:cloud",
    base_url="https://ollama.com",
    client_kwargs={"headers": {"Authorization": "Bearer " + os.environ["OLLAMA_API_KEY"]}},
)
print(ping.invoke("回一個字：好").content)

---

### A1・把圖片讀進來，變成模型看得懂的格式（base64）

模型不能直接吃 PNG 檔，要先把圖片轉成一長串**文字**（base64 編碼）再送出去。

**預期輸出：** 一個很大的長度數字 + 一段看起來像亂碼的文字開頭。

In [ ]:
import base64

# 先讀台積電損益表這張圖（B 段會用它）
image_bytes = open("data/fig2.png", "rb").read()
# TODO：用 base64 的『編碼』方法，把圖片 bytes 變成一長串文字（後面的 .decode() 會幫你把 bytes 轉成 str）
image_b64 = base64.____(image_bytes).decode()

print("這張圖變成一長串文字，總長度：", len(image_b64))
print("開頭 80 個字：", image_b64[:80])

### A2・把「文字問題 ＋ 圖片」包成一則多模態訊息

多模態訊息就是一則訊息裡**同時裝文字和圖片**兩塊。

**預期輸出：** 這則訊息有兩塊 —— 一塊 `text`、一塊 `image_url`。

In [ ]:
from langchain_core.messages import HumanMessage

question = "這張圖在講什麼？用一句話說。"
message = HumanMessage(content=[
    {"type": "text", "text": question},
    # TODO：接上 A1 算出來的那串 base64 文字（它存在哪個變數裡？）
    {"type": "image_url", "image_url": "data:image/png;base64," + ____},
])

print("這則訊息有兩塊：")
for part in message.content:
    print("-", part["type"])

### A3・建立兩個模型物件（開源 minimax｜商用 gemini）

**預期輸出：** 兩行，各自說明是哪一顆、跑在哪。

In [ ]:
from langchain_ollama import ChatOllama
from langchain_google_genai import ChatGoogleGenerativeAI

# (1) 開源模型 minimax —— 跑 Ollama 雲端，要帶 key
minimax = ChatOllama(
    model="minimax-m3:cloud",
    base_url="https://ollama.com",
    client_kwargs={"headers": {"Authorization": "Bearer " + os.environ["OLLAMA_API_KEY"]}},
)

# (2) 商用模型 gemini —— 自動讀環境變數裡的 GOOGLE_API_KEY
gemini = ChatGoogleGenerativeAI(model="gemini-3.5-flash")

# 💡 只換一個 class 就換一家供應商——這正是 Lab 2 看過的「換模型不用重寫程式」
print("開源：", minimax.model, "→ Ollama 雲端")
print("商用：gemini-3.5-flash → Google 雲端")

### A4・定義一個小工具：把「問題＋圖」送給任一模型

B、C 段要問好幾次，先包成一個函式，之後只要換模型和問題就好。

**預期輸出：** 一行「工具準備好了」。

In [ ]:
def read_image(model, question, img_b64, is_gemini=False):
    message = HumanMessage(content=[
        {"type": "text", "text": question},
        # TODO：接上這個函式收到的圖片參數（看 def 括號裡收到的那個名字，不是外面那個全域變數）
        {"type": "image_url", "image_url": "data:image/png;base64," + ____},
    ])
    reply = model.invoke([message])
    if is_gemini:
        answer = reply.text        # Gemini 用 .text 取乾淨文字
    else:
        answer = reply.content     # minimax(ChatOllama) 用 .content
    return answer, reply.usage_metadata

print("工具準備好了")

---

### B1・minimax 讀台積電損益表

先用一張**讀得準**的密集財報表建立信心：8 季 × 幾十列的小字，問它三個數字。

**預期輸出：** 2026.1Q 營業毛利 751,295、營業利益最高 2026.1Q 658,966、毛利率約 66.25%。

In [ ]:
# TODO：呼叫你在 A4 定義的那個讀圖小工具
ans_minimax, usage_minimax = ____(minimax, '這是台積電(2330)的綜合損益表季表(單位:百萬)。請回答：(1)2026.1Q的營業毛利是多少？(2)這8季裡營業利益最高的是哪一季、多少？(3)2026.1Q的營業收入淨額是多少？用它和營業毛利算出2026.1Q的毛利率(%)。', image_b64)
print(ans_minimax)

### B2・Gemini 讀同一張損益表

In [ ]:
# TODO：一樣呼叫那個讀圖小工具，這次多帶一個 is_gemini=True（Gemini 取字用 .text）
ans_gemini, usage_gemini = ____(gemini, '這是台積電(2330)的綜合損益表季表(單位:百萬)。請回答：(1)2026.1Q的營業毛利是多少？(2)這8季裡營業利益最高的是哪一季、多少？(3)2026.1Q的營業收入淨額是多少？用它和營業毛利算出2026.1Q的毛利率(%)。', image_b64, is_gemini=True)
print(ans_gemini)

### 🔑 兩顆都把密集表格讀對了

8 季 × 幾十列的小字，連非單調的陷阱（2025.1Q 反而低於 2024.4Q）都沒讀錯。密集財報表格難不倒它們。

In [ ]:
# 同一份工作，看兩顆各燒多少 token
print("minimax token：", usage_minimax)
print("gemini  token：", usage_gemini)

# 💡 gemini 的 output_token_details 裡有一大塊 reasoning（看不見的「思考」），照樣算錢；
#    minimax 這邊沒有這塊。答案品質差不多，成本結構卻不同——貴不一定值得，要看場景。

---

### C1・換一張更難的圖：先進光 K 線 ＋ 資券

這張比較亂：K 線 ＋ 融資 ＋ 融券 ＋ 借券五個子圖疊在一起，而且藏了一個**陷阱**。

**預期輸出：** K 線圖也轉成 base64，印出長度。

In [ ]:
kline_bytes = open("data/fig1.png", "rb").read()
kline_b64 = base64.b64encode(kline_bytes).decode()
print("K 線圖轉好了，長度：", len(kline_b64))

### C2・兩顆一起讀 K 線圖

In [ ]:
ans_k_minimax, _ = read_image(
    minimax,
    """
    這是先進光(3362)的資券進出行情圖。請回答：
    (1)最新收盤價、開盤、最高、最低各多少？
    (2)股價從年初到現在的整體走勢？大約什麼時候開始大漲、漲到多少？
    (3)融資餘額和融券餘額目前各多少張？
    """,
    kline_b64
)
print(ans_k_minimax)

In [ ]:
ans_k_gemini, _ = read_image(
    gemini,
    """
    這是先進光(3362)的資券進出行情圖。請回答：
    (1)最新收盤價、開盤、最高、最低各多少？
    (2)股價從年初到現在的整體走勢？大約什麼時候開始大漲、漲到多少？
    (3)融資餘額和融券餘額目前各多少張？
    """,
    kline_b64, is_gemini=True
)
print(ans_k_gemini)

### 🔑 這次出現差距了：其中一顆把數字讀歪了

圖左上角「2026-02-02 收 106」是圖**最左端（2 月）** 那根 K 棒的報價；股價是從這裡（2 月）**往右一路漲到 7 月的 225**，圖尾現在約 150～175。

- **minimax**：走勢大致抓到了，卻冒出一句「**收盤價 106 元（2 月初）較高點回檔約 53%**」——把 2 月的 106 當成 7 月高點**之後**的修正低點。但 2 月在 7 月**之前**，根本不可能從 7 月的高點「回檔」到 2 月的價位。**時間軸被它讀反了。**
- **gemini**：老實讀成「2 月盤整（100～125）→ 5 月底大漲到 225 → 7 月後回到 150～175」，把 106 當成 2 月那天的報價，沒有搞反時序。

⚠️ 多模態很強，但圖上有**多個/不一致的數字**（左端的舊報價 vs 右端的最新價）時會被誤導。**關鍵數字一定要自己回頭核對圖，別照單全收**——下面小作業會看到：只要把問題問得夠具體（點出日期），連 minimax 都能答對。

### 📝 小作業：換上你自己的圖，問你自己的問題

把你的一張圖（財報截圖、K 線、儀表板、甚至隨手拍的照片）**丟進 `data/`**，用下面那格先定義好的 `read_image_from_file`（直接吃**檔名**、不用自己轉 base64）分析它。

<details><summary>📖 做完再看：參考解（參考解不只一種）</summary>

```python
# 把 K 線那張的陷阱點出來問：問得越具體，越不容易被讀歪
my_question = "圖左上角那個「收 106」是最新的股價嗎？請先看清楚它旁邊的日期，再回答這是圖上哪一段的價格。"
my_answer, _ = read_image_from_file(minimax, my_question, "data/fig1.png")
print(my_answer)
```

把問題聚焦到「日期」，minimax 就會發現 106 是 2 月的報價、不是最新價。
**結論：問得越具體，讀圖越不容易被誤導。**
</details>

In [ ]:
# 這個小工具直接吃『檔名』：自動幫你讀檔 → 轉 base64 → 呼叫上面的 read_image
def read_image_from_file(model, question, image_path, is_gemini=False):
    image_bytes = open(image_path, "rb").read()
    img_b64 = base64.b64encode(image_bytes).decode()   # 💡 跟 A1 同一招
    return read_image(model, question, img_b64, is_gemini)   # 💡 再交給 A4 的工具

print("工具準備好了：read_image_from_file 直接吃檔名")

In [ ]:
# 📝 你的答案：把你的圖丟進 data/，換掉下面的檔名和問題
my_image = "data/fig1.png"   # 換成你丟進 data/ 的圖檔名（先用 K 線這張也行）
# TODO：寫一個你想問這張圖的問題（越具體越好——問題越明確，模型越不會讀歪）
my_question = ____
my_answer, _ = read_image_from_file(minimax, my_question, my_image)
print(my_answer)

---

### 📌 小結：多模態能讀非純文字的財報素材

K 線圖、損益表截圖、儀表板、PDF，模型都能直接讀——不用先把圖上的字一個個打成文字。

- **開源 minimax 讀圖不輸商用 gemini**，成本卻低不少（gemini 燒大量看不見的 reasoning token）。
- **但讀圖不保證 100% 正確**（K 線那張就露餡了）→ 金融上**關鍵數字要人工核對**。
- **隱私提醒**：圖送給 Gemini＝資料送到 Google 的雲；機敏財報就得走 Lab 2 的自架開源路線。

> 下一段（區塊 5）會把「讀圖」接進 RAG，做**多模態 RAG**——讓模型連圖表一起檢索、一起回答。

---

In [ ]:
# 平常不用跑這格。雲端不通（401 / 連線失敗 / 額度用完）時的 Backup：
# minimax 掛了 → 把讀圖模型換成本機有 vision 的模型（需先 ollama pull）；
# gemini 掛了 → 這段對照就先看 minimax 一顆的結果，流程不會斷。
#
# from langchain_ollama import ChatOllama
# minimax = ChatOllama(model="你本機的vision模型")   # 例如 pull 一顆能讀圖的本機模型